### 환경 설정

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import koreanize_matplotlib
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')

from src.preprocess import preprocess
from src.features import add_features

plt.rcParams['figure.dpi'] = 120

### 데이터 로드

In [2]:
DATA_PATH = '../data/raw/'

# 경로 설정 (Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# ROOT = '/content/drive/MyDrive/PREGNANCY-ML-MODELS'
# DATA_PATH = f'{ROOT}/data/raw/'

train = pd.read_csv(DATA_PATH + 'train.csv')
test = pd.read_csv(DATA_PATH + 'test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape : {test.shape}')

Train shape: (256351, 69)
Test shape : (90067, 68)


### 전처리 적용

In [3]:
train_processed = preprocess(train)
test_processed = preprocess(test)

print(f'Train shape: {train_processed.shape}')
print(f'Test shape : {test_processed.shape}')


Train shape: (256351, 66)
Test shape : (90067, 65)


### 파생 변수 추가

In [4]:
train_featured = add_features(train_processed)
test_featured = add_features(test_processed)

print(f'Train shape: {train_featured.shape}')
print(f'Test shape : {test_featured.shape}')

# 새로 추가된 파생 변수 확인
original_cols = set(train_processed.columns)
new_cols = set(train_featured.columns) - original_cols
print(f'\n추가된 파생 변수 ({len(new_cols)}개):')
print(sorted(new_cols))

Train shape: (256351, 77)
Test shape : (90067, 76)

추가된 파생 변수 (11개):
['난자_수정률', '미세주입_성공률', '배아_이식률', '배아_저장률', '신선_난자_활용률', '연령_x_생성배아수', '연령_x_시술횟수', '연령_x_이식배아수', '연령_그룹', '임신_성공률_이력', '출산_성공률_이력']


### 파생 변수 상관관계 확인

In [5]:
NEW_COLS = [
    '배아_이식률', '배아_저장률', '난자_수정률', '미세주입_성공률', '신선_난자_활용률',
    '연령_그룹', '연령_x_시술횟수', '연령_x_이식배아수', '연령_x_생성배아수',
    '임신_성공률_이력', '출산_성공률_이력'
]

corr = train_featured[NEW_COLS + ['임신 성공 여부']].corr()['임신 성공 여부'].drop('임신 성공 여부').sort_values(key=abs, ascending=False)

print(corr)

print(train_featured[['연령_그룹', '임신 성공 여부']].corr()['임신 성공 여부'])

연령_그룹        -0.148657
연령_x_시술횟수    -0.100299
미세주입_성공률      0.060954
연령_x_이식배아수   -0.057372
신선_난자_활용률     0.052492
난자_수정률        0.052240
배아_저장률        0.031986
배아_이식률       -0.024661
임신_성공률_이력     0.014683
출산_성공률_이력     0.008804
연령_x_생성배아수   -0.008119
Name: 임신 성공 여부, dtype: float64
연령_그룹      -0.148657
임신 성공 여부    1.000000
Name: 임신 성공 여부, dtype: float64


In [14]:
feature_map = {
    '연령_그룹':        ['시술 당시 나이'],
    '연령_x_시술횟수':   ['시술 당시 나이', '총 시술 횟수'],
    '연령_x_이식배아수':  ['시술 당시 나이', '이식된 배아 수'],
    '연령_x_생성배아수':  ['시술 당시 나이', '총 생성 배아 수'],
    '미세주입_성공률':    ['미세주입에서 생성된 배아 수', '미세주입된 난자 수'],
    '배아_이식률':       ['이식된 배아 수', '총 생성 배아 수'],
    '배아_저장률':       ['저장된 배아 수', '총 생성 배아 수'],
    '난자_수정률':       ['혼합된 난자 수', '수집된 신선 난자 수'],
    '신선_난자_활용률':   ['혼합된 난자 수', '수집된 신선 난자 수', '해동 난자 수'],
    '임신_성공률_이력':   ['총 임신 횟수', '총 시술 횟수'],
    '출산_성공률_이력':   ['총 출산 횟수', '총 임신 횟수'],
}

all_corr = train_featured.corr(numeric_only=True)['임신 성공 여부'].abs()

rows = []
for derived_col, orig_cols in feature_map.items():
    derived_corr = all_corr.get(derived_col, None)
    orig_corrs = {c: round(all_corr.get(c, float('nan')), 3) for c in orig_cols}
    rows.append({
        '파생 변수': derived_col,
        '파생 상관계수': round(derived_corr, 3),
        '사용된 원본 변수': ', '.join(orig_cols),
        '원본 상관계수': ', '.join([f'{c}: {v}' for c, v in orig_corrs.items()])
    })

result_df = pd.DataFrame(rows).sort_values('파생 상관계수', ascending=False)
print(result_df.to_string(index=False))

     파생 변수  파생 상관계수                      사용된 원본 변수                                             원본 상관계수
     연령_그룹    0.149                       시술 당시 나이                                       시술 당시 나이: nan
 연령_x_시술횟수    0.100              시술 당시 나이, 총 시술 횟수                       시술 당시 나이: nan, 총 시술 횟수: 0.059
  미세주입_성공률    0.061    미세주입에서 생성된 배아 수, 미세주입된 난자 수             미세주입에서 생성된 배아 수: 0.09, 미세주입된 난자 수: 0.07
연령_x_이식배아수    0.057             시술 당시 나이, 이식된 배아 수                      시술 당시 나이: nan, 이식된 배아 수: 0.157
    난자_수정률    0.052          혼합된 난자 수, 수집된 신선 난자 수                 혼합된 난자 수: 0.116, 수집된 신선 난자 수: 0.083
 신선_난자_활용률    0.052 혼합된 난자 수, 수집된 신선 난자 수, 해동 난자 수 혼합된 난자 수: 0.116, 수집된 신선 난자 수: 0.083, 해동 난자 수: 0.002
    배아_저장률    0.032            저장된 배아 수, 총 생성 배아 수                   저장된 배아 수: 0.036, 총 생성 배아 수: 0.146
    배아_이식률    0.025            이식된 배아 수, 총 생성 배아 수                   이식된 배아 수: 0.157, 총 생성 배아 수: 0.146
 임신_성공률_이력    0.015               총 임신 횟수, 총 시술 횟수                      총